# 04 – Results & Visualisation

This notebook demonstrates:
1. Interactive 3-D Plotly trajectory comparison
2. Per-component (Δx, Δy, Δz) error over time
3. RMSE / MAE comparison bar chart
4. Animated GIF export

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import torch
import numpy as np

from src.data_loader import (
    load_tle_file, build_dataset, create_windows, split_and_normalize,
    WINDOW_SIZE, STEP_MIN, HOURS
)
from src.model import (
    OrbitalLSTM, predict_lstm, RandomForestPredictor, compute_metrics
)
from src.visualization import (
    plot_trajectory_3d, plot_error_components,
    plot_rmse_comparison, create_trajectory_gif
)

## 4.1 Load pre-trained model and run inference

In [ ]:
tle_list = load_tle_file('../data/starlink_tle.txt')
records  = build_dataset(tle_list, hours=HOURS, step_min=STEP_MIN)
X, y     = create_windows(records, window_size=WINDOW_SIZE)
X_train, y_train, X_test, y_test, x_sc, y_sc = split_and_normalize(X, y)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = OrbitalLSTM()

weights_path = '../lstm.pt'
if os.path.isfile(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print('Loaded pre-trained weights.')
else:
    from src.model import train_lstm
    print('Training from scratch …')
    train_lstm(model, X_train, y_train, epochs=50, device=device)

y_pred_n  = predict_lstm(model, X_test, device=device)
y_pred_km = y_sc.inverse_transform(y_pred_n)
y_true_km = y_sc.inverse_transform(y_test)

lstm_m = compute_metrics(y_true_km, y_pred_km)
print(f'LSTM  RMSE={lstm_m["RMSE_km"]:.3f} km   MAE={lstm_m["MAE_km"]:.3f} km')

## 4.2 Random Forest metrics

In [ ]:
rf = RandomForestPredictor()
rf.fit(X_train, y_train)
y_rf_km = y_sc.inverse_transform(rf.predict(X_test))
rf_m    = compute_metrics(y_true_km, y_rf_km)
print(f'RF    RMSE={rf_m["RMSE_km"]:.3f} km   MAE={rf_m["MAE_km"]:.3f} km')

## 4.3 Comparison table

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'Model': ['LSTM (2L×64)', 'Random Forest'],
    'RMSE (km)': [lstm_m['RMSE_km'], rf_m['RMSE_km']],
    'MAE (km)':  [lstm_m['MAE_km'],  rf_m['MAE_km']],
})
display(df.set_index('Model').round(3))

## 4.4 Interactive 3-D trajectory

In [ ]:
fig = plot_trajectory_3d(y_true_km[:288], y_pred_km[:288])
fig.show()

## 4.5 Per-component error

In [ ]:
plot_error_components(y_true_km, y_pred_km, output_path='../results/error_components.png')

## 4.6 RMSE comparison bar chart

In [ ]:
plot_rmse_comparison(
    {'LSTM': lstm_m, 'Random Forest': rf_m},
    output_path='../results/rmse_comparison.png'
)

## 4.7 Animated GIF (optional – can take ~1 min)

In [ ]:
# Uncomment to generate the GIF (requires imageio)
# create_trajectory_gif(y_true_km, y_pred_km, output_path='../results/trajectory_3d.gif')